# Simplifying Python Applications Through Functions and Modules

---

## Table of Contents

1. [Module 1 — Functions](#module-1)
2. [Module 2 — Modules](#module-2)
3. [Module 3 — Packages](#module-3)
4. [Module 4 — Decorators and Closures](#module-4)

---

## Getting Started with the Course Repo

The course project lives at:
https://github.com/andy489/python-essentials-path/tree/main/15-functions-modules/course-python-functions-modules

### Clone and set up

```bash
# 1. Clone the repo
git clone https://github.com/andy489/python-essentials-path
cd python-essentials-path/15-functions-modules/course-python-functions-modules

# 2. Set up the virtual environment (requires uv — install via: pip install uv)
uv sync

# 3. Activate the environment
source .venv/bin/activate          # macOS / Linux
# .venv\Scripts\activate           # Windows

# 4. Install the package in editable mode (code changes take effect immediately)
uv pip install --editable .
```

### Run the tools

```bash
wcl https://github.com/andy489/python-essentials-path       # clone a repo
open_repo https://github.com/andy489/python-essentials-path  # clone + open in VS Code
```

### Run the tests

```bash
pytest              # run all tests once
ptw                 # watch mode — re-runs on file changes
ptw pathz           # watch only the pathz package
```

# Module 1 — Functions

Functions are the building blocks of reusable, readable Python programs. This module covers everything from the basics to advanced parameter patterns.

```
┌─────────────────────────────────────────────────────────┐
│                     FUNCTION ANATOMY                    │
│                                                         │
│   def function_name(param1, param2, *args, **kwargs):   │
│       │              │       │       │       │           │
│       │              │       │       │       └── keyword │
│       │              │       │       │           dict    │
│       │              │       │       └────────── var.    │
│       │              │       │                  positional│
│       │              │       └────────────────── positional│
│       │              └────────────────────────── positional│
│       └───────────────────────────────────────── name    │
│                                                         │
│       """docstring"""   ◄── optional documentation      │
│       body                                              │
│       return value     ◄── None if omitted              │
└─────────────────────────────────────────────────────────┘
```

## 1.1 Defining and Calling Functions

In [1]:
# Basic function definition
def greet(name):
    """Return a greeting string."""
    return f"Hello, {name}!"

print(greet("Wes"))

# Functions are first-class objects
say_hello = greet
print(say_hello("Python"))
print(type(greet))

Hello, Wes!
Hello, Python!
<class 'function'>


## 1.2 Parameters and Arguments

```
PARAMETER TYPES
═══════════════════════════════════════════════════════════

 positional  │  keyword-only  │  var-positional │ var-keyword
─────────────┼────────────────┼─────────────────┼────────────
 def f(a, b) │ def f(*, k)    │ def f(*args)    │ def f(**kw)
 f(1, 2)     │ f(k=3)         │ f(1,2,3,...)    │ f(x=1,y=2)
═══════════════════════════════════════════════════════════
```

In [2]:
# Default parameter values
def connect(host, port=5432, timeout=30):
    return f"Connecting to {host}:{port} (timeout={timeout}s)"

print(connect("localhost"))                  # uses defaults
print(connect("db.prod", port=5433))        # override one default
print(connect("db.prod", 5433, timeout=10)) # positional + keyword

# *args — variable positional arguments
def total(*numbers):
    return sum(numbers)

print(total(1, 2, 3, 4, 5))  # 15

# **kwargs — variable keyword arguments
def describe(**attrs):
    for key, value in attrs.items():
        print(f"  {key}: {value}")

describe(name="Python", version=3.12, type="language")

Connecting to localhost:5432 (timeout=30s)
Connecting to db.prod:5433 (timeout=30s)
Connecting to db.prod:5433 (timeout=10s)
15
  name: Python
  version: 3.12
  type: language


In [3]:
# Keyword-only parameters (after *)
def process(data, *, verbose=False, dry_run=False):
    """verbose and dry_run MUST be passed as keywords."""
    if verbose:
        print(f"Processing {len(data)} items")
    if not dry_run:
        return [item.upper() for item in data]
    return []

result = process(["a", "b", "c"], verbose=True)
print(result)

# Positional-only parameters (before /)
def divide(a, b, /):
    """a and b MUST be positional — cannot be named."""
    return a / b

print(divide(10, 2))   # OK
# divide(a=10, b=2)    # TypeError

Processing 3 items
['A', 'B', 'C']
5.0


## 1.3 Return Values

In [4]:
# Multiple return values via tuple unpacking
def min_max(numbers):
    return min(numbers), max(numbers)

lo, hi = min_max([3, 1, 4, 1, 5, 9, 2, 6])
print(f"min={lo}, max={hi}")

# Returning None explicitly vs implicitly
def log(msg):
    print(f"[LOG] {msg}")  # returns None implicitly

result = log("hello")
print(f"log() returned: {result!r}")

min=1, max=9
[LOG] hello
log() returned: None


## 1.4 Type Hints

Type hints don't enforce at runtime but improve readability and enable static analysis tools (mypy, pyright).

In [ ]:
from typing import Optional

def parse_url(url: str) -> tuple[str, str]:
    """Parse a git URL, return (repo_dir, clone_from)."""
    # simplified example
    parts = url.split("/")
    return "/".join(parts[:3]), url

def find_user(user_id: int) -> Optional[str]:
    """Return user name or None if not found."""
    users = {1: "Alice", 2: "Bob"}
    return users.get(user_id)

path, clone = parse_url("https://github.com/andy489/python-essentials-path")
print(f"path={path}")
print(find_user(1))
print(find_user(99))

## 1.5 Lambda Functions

In [6]:
# lambda: anonymous function for short one-liners
square = lambda x: x ** 2
print(square(5))  # 25

# Common use: as a key function
repos = [
    {"name": "linux", "stars": 180000},
    {"name": "cpython", "stars": 62000},
    {"name": "numpy", "stars": 27000},
]

sorted_repos = sorted(repos, key=lambda r: r["stars"], reverse=True)
for repo in sorted_repos:
    print(f"  {repo['name']}: ⭐ {repo['stars']:,}")

25
  linux: ⭐ 180,000
  cpython: ⭐ 62,000
  numpy: ⭐ 27,000


## 1.6 Scope and the LEGB Rule

```
LEGB SCOPE LOOKUP ORDER
════════════════════════════════════════

  L — Local      (inside current function)
       ↓
  E — Enclosing  (outer function for closures)
       ↓
  G — Global     (module level)
       ↓
  B — Built-in   (len, print, int, ...)

  Python searches L → E → G → B and stops
  at the first match.
════════════════════════════════════════
```

In [7]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print(f"inner sees: {x}")      # local
    inner()
    print(f"outer sees: {x}")          # enclosing

outer()
print(f"global sees: {x}")             # global

# global and nonlocal keywords
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
print(f"counter = {counter}")  # 2

inner sees: local
outer sees: enclosing
global sees: global
counter = 2


---
# Module 2 — Modules

A **module** is a `.py` file that can be imported. Modules let you split code into logical units and reuse them across projects.

```
MODULE IMPORT FLOW
═══════════════════════════════════════════════════════════

  import mymodule
       │
       ├─► 1. Check sys.modules cache (already imported?)
       │         YES → return cached module object
       │         NO  ↓
       ├─► 2. Find the file
       │         sys.path searched in order:
       │           [script dir, PYTHONPATH, stdlib, site-packages]
       ├─► 3. Create module object
       ├─► 4. Add to sys.modules
       └─► 5. Execute module code (top-level statements run ONCE)
═══════════════════════════════════════════════════════════
```

## 2.1 Import Styles

In [8]:
# Style 1: import module
import os
print(os.path.join("/home", "user", "file.txt"))

# Style 2: from module import name
from os.path import join, exists, expanduser
print(join("/home", "user", "file.txt"))

# Style 3: alias
import os.path as osp
print(osp.abspath("."))

# Style 4: import all (use sparingly — pollutes namespace)
# from os.path import *

/home/user/file.txt
/home/user/file.txt
/Users/I777690/repos/python-essentials-path/15-functions-modules


## 2.2 `__name__` and the `if __name__ == "__main__":` Guard

```
EXECUTION MODES
═══════════════════════════════════════════════════

  Run directly:    python3 mymodule.py
                     __name__ == "__main__"  ← True

  Imported:        import mymodule
                     __name__ == "mymodule"  ← True

  Guard pattern:
    if __name__ == "__main__":
        main()     ← only runs when executed directly
═══════════════════════════════════════════════════
```

In [ ]:
# This is the pattern used in reponow/wcl.py and reponow/opener.py

# --- wcl.py (simplified) ---
import argparse
import os

def clone_repo(url: str, dry_run: bool = False) -> None:
    """Clone a repo to a consistent local path."""
    repo_dir = os.path.join(os.path.expanduser("~/repos"), "github", "user", url.split("/")[-1])
    print(f"Would clone {url} → {repo_dir}" if dry_run else f"Cloning {url}")

def main():
    parser = argparse.ArgumentParser(description="(w)es (cl)one", prog="wcl")
    parser.add_argument("url", type=str)
    parser.add_argument("--dry-run", action="store_true")
    args = parser.parse_args(["https://github.com/andy489/python-essentials-path"])  # example args for notebook
    # clone_repo(args.url, args.dry_run)

# The guard:
if __name__ == "__main__":
    main()

print(f"Current __name__: {__name__}")

## 2.3 Module Attributes and Introspection

In [ ]:
import os
import sys

# Every module has these dunder attributes
print(f"os.__name__  : {os.__name__}")
print(f"os.__file__  : {os.__file__}")
print(f"os.__doc__[:50]: {os.__doc__[:50]!r}")

# sys.modules: cache of all imported modules
print(f"\nModules starting with 'os': {[k for k in sys.modules if k.startswith('os')]}")

# sys.path: where Python searches for modules
print("\nsys.path entries:")
for p in sys.path[:5]:
    print(f"  {p}")

## 2.4 The `reponow` Module — Real-World Example

This is the course demo project. The `parser.py` module demonstrates clean function decomposition.

In [ ]:
# Illustrating the parse_url logic from reponow/parser.py
import re
from urllib.parse import urlparse

HOST_ALIASES = {
    "github.com": "github",
    "gitlab.com": "gitlab",
    "bitbucket.org": "bitbucket",
}

def parse_url(url: str) -> tuple[str, str]:
    """Parse a git URL to (local_path, clone_url)."""
    url = url.strip().removesuffix(".git")

    if url.startswith("git@"):  # SSH: git@github.com:user/repo
        match = re.match(r"git@([^:]+):(.+)", url)
        domain, path = match.groups()
    elif url.startswith("https://"):  # HTTPS
        parsed = urlparse(url)
        domain = parsed.netloc
        path = parsed.path.strip("/")
        # strip /blob/ and /tree/ navigation paths
        path = re.sub(r"/(blob|tree).*", "", path)
    elif "/" in url:  # shorthand: user/repo
        domain, path = "github.com", url
    else:  # shorthand: repo (assumes andy489)
        domain, path = "github.com", f"andy489/{url}"

    host = HOST_ALIASES.get(domain, domain)
    import os
    local_path = os.path.expanduser(f"~/repos/{host}/{path}")
    return local_path, url

# Test cases
test_cases = [
    "https://github.com/andy489/python-essentials-path",
    "git@github.com:andy489/python-essentials-path",
    "python/cpython",
    "python-essentials-path",
]

for url in test_cases:
    path, clone = parse_url(url)
    print(f"{url!r:50} → {path}")

---
# Module 3 — Packages

A **package** is a directory with an `__init__.py` file that groups related modules together.

```
PACKAGE STRUCTURE
══════════════════════════════════════════════════════

  reponow/                   ← package directory
  ├── __init__.py            ← marks it as a package
  │                            (can export public API)
  ├── __main__.py            ← python -m reponow
  ├── parser.py              ← submodule
  ├── wcl.py                 ← submodule  
  └── opener.py              ← submodule

  pathz/
  ├── __init__.py            ← exposes join, exists, abspath
  ├── join_test.py
  └── parameterize_test.py

  Imports:
    from reponow.parser import parse_url   ← submodule import
    from reponow import parse_url          ← via __init__.py
    import pathz as path                   ← package as namespace
══════════════════════════════════════════════════════
```

## 3.1 `__init__.py` — The Package Gateway

In [ ]:
# What reponow/__init__.py does:
#
#   from .parser import parse_url
#
# This lets users do:
#   from reponow import parse_url
# instead of:
#   from reponow.parser import parse_url
#
# __init__.py controls the PUBLIC API of your package.

# Demonstrating relative imports (dot notation)
# In parser.py:  from .parser import parse_url  (relative)
# In parser.py:  from reponow.parser import ...  (absolute)

# Relative imports explained:
print("""
Relative import syntax:

  from .sibling import something     # same package
  from ..parent import something     # one level up
  from ..sibling import something    # up then across
""")

## 3.2 `__main__.py` — Running as a Script

In [ ]:
# python -m reponow  →  runs reponow/__main__.py
#
# Three equivalent ways to run the wcl command:
#   python3 reponow/wcl.py
#   python3 -m reponow.wcl
#   python3 -m reponow      (via __main__.py)

print("""
The -m flag difference:

  python3 reponow/wcl.py     sys.path[0] = reponow/
  python3 -m reponow.wcl     sys.path[0] = project root

Using -m is preferred: consistent sys.path,
works from any working directory.
""")

## 3.3 Packaging for Distribution — `pyproject.toml`

In [ ]:
# The course project uses a modern pyproject.toml (PEP 517/518)
# Key sections:

pyproject_toml_example = """
[project]
name = "reponow"
version = "0.1.0"
description = "Clone repos to consistent local paths"
dependencies = ["rich"]

[project.scripts]
wcl = "reponow.wcl:main"       # entry point: package.module:function
opener = "reponow.opener:main"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
"""

print(pyproject_toml_example)

print("""
Installation commands:

  uv sync                   # install with uv (fast)
  pip install .             # install from current directory
  pip install -e .          # editable install (code changes take effect immediately)
  pipx install reponow      # install as isolated CLI tool
""")

## 3.4 The `pathz` Package — Wrapping `os.path`

The `pathz` package from the course demonstrates using decorators to extend `os.path` functions with tilde expansion.

In [ ]:
import os

# The pathz package wraps os.path functions
# so that ~/path automatically expands the tilde.

def expand_tilde_first_and_then(path_func):
    """Decorator factory: expand ~ before passing to path_func."""
    def wrapped(first_component, *parts):
        return path_func(os.path.expanduser(first_component), *parts)
    return wrapped

# Wrapping stdlib functions with our decorator
join = expand_tilde_first_and_then(os.path.join)
abspath = expand_tilde_first_and_then(os.path.abspath)

# Now ~ expansion is automatic:
print(join("~/repos", "github", "user", "repo"))
print(abspath("~/"))

## 3.5 Testing Packages — `pytest` + Parametrize

In [ ]:
# The course uses pytest with parametrize for clean test tables
# From pathz/parameterize_test.py:

# @pytest.mark.parametrize("input_url,expected_path", [
#     pytest.param(
#         "https://github.com/andy489/python-essentials-path",
#         f"{HOME}/repos/github/andy489/python-essentials-path",
#         id="github_https"
#     ),
#     pytest.param(
#         "git@github.com:andy489/python-essentials-path",
#         f"{HOME}/repos/github/andy489/python-essentials-path",
#         id="github_ssh"
#     ),
#     # ... more cases
# ])
# def test_parsing_repo_dir(input_url, expected_path):
#     repo_dir, _ = parse_url(input_url)
#     assert repo_dir == expected_path

# Let's run equivalent tests inline:
import os
HOME = os.path.expanduser("~")

test_cases_parse = [
    ("https://github.com/andy489/python-essentials-path", f"{HOME}/repos/github/andy489/python-essentials-path"),
    ("git@github.com:andy489/python-essentials-path",     f"{HOME}/repos/github/andy489/python-essentials-path"),
    ("python/cpython",                                     f"{HOME}/repos/github/python/cpython"),
]

passed = 0
for url, expected in test_cases_parse:
    result, _ = parse_url(url)
    status = "PASS" if result == expected else f"FAIL (got {result})"
    print(f"  [{status}] {url}")
    if result == expected:
        passed += 1

print(f"\n{passed}/{len(test_cases_parse)} tests passed")

---
# Module 4 — Decorators and Closures

Decorators and closures are two of Python's most powerful higher-order function features. The `pathz` package in the course is a real-world showcase of both.

```
CLOSURES — How they work
══════════════════════════════════════════════════════

  def outer(x):         ← x is a "free variable"
      def inner(y):     ← inner "closes over" x
          return x + y  ← x lives in inner.__closure__
      return inner      ← returns the function object

  add5 = outer(5)       ← x=5 is captured
  add5(3)               → 8   (x remembered!)
  add5(10)              → 15

  Key: the enclosing scope's variable lives as long
  as the inner function is alive.
══════════════════════════════════════════════════════
```

## 4.1 Closures

In [ ]:
# Basic closure
def make_multiplier(factor):
    def multiply(x):
        return x * factor  # factor is captured from enclosing scope
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15

# Inspect what's captured
print(f"\ndouble.__closure__: {double.__closure__}")
print(f"captured value: {double.__closure__[0].cell_contents}")

# nonlocal: modify captured variable
def make_counter(start=0):
    count = start
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

counter = make_counter()
print(counter())  # 1
print(counter())  # 2
print(counter())  # 3

## 4.2 Decorators — The Pattern

```
DECORATOR SYNTAX SUGAR
══════════════════════════════════════════════════════

  @my_decorator          EQUIVALENT TO:
  def my_func():         def my_func():
      ...                    ...
                         my_func = my_decorator(my_func)

  A decorator is a callable that:
    1. Takes a function as argument
    2. Returns a new function (wrapper)
    3. The wrapper calls the original + adds behavior

  Stacking decorators:
    @decorator_A          Applied bottom-up:
    @decorator_B          func = decorator_A(decorator_B(func))
    def func(): ...
══════════════════════════════════════════════════════
```

In [ ]:
import functools
import time

# Simple decorator
def timer(func):
    @functools.wraps(func)  # preserve func metadata
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

result = slow_sum(1_000_000)
print(f"Result: {result:,}")
print(f"Function name preserved: {slow_sum.__name__}")

## 4.3 The `countit` Decorator — From the Course

In [ ]:
# From pathz/__init__.py (refactored — now uses functools.wraps)
import functools

def countit(fn):
    """Decorator: count and log every call to fn."""
    counter = 0

    @functools.wraps(fn)  # preserves __name__, __doc__ on the wrapper
    def wrapper(*args, **kwargs):
        nonlocal counter
        counter += 1
        result = fn(*args, **kwargs)
        print(f"count: {counter} {fn.__name__}")
        return result

    return wrapper

@countit
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))
print(greet("Bob"))
print(greet("Charlie"))
print(f"__name__ preserved: {greet.__name__}")

## 4.4 Decorator Factories (Parametrized Decorators)

```
DECORATOR FACTORY PATTERN
══════════════════════════════════════════════════════

  @retry(times=3)           ← factory called with args
  def fetch(): ...             returns a decorator
                               decorator wraps fetch

  Three levels:
    def retry(times):          ← 1. factory (takes config)
        def decorator(func):   ← 2. decorator (takes func)
            def wrapper(...):  ← 3. wrapper (called in place of func)
                ...
            return wrapper
        return decorator
══════════════════════════════════════════════════════
```

In [ ]:
import functools
import random

def retry(times=3, exceptions=(Exception,)):
    """Decorator factory: retry up to `times` on specified exceptions."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"Attempt {attempt}/{times} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator

# Usage
attempt_count = 0

@retry(times=3)
def flaky_operation():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ConnectionError("Network error")
    return "Success!"

result = flaky_operation()
print(f"Final result: {result}")

## 4.5 Stacking Decorators — From the Course

The `pathz` package stacks `@expand_tilde2` and `@countit` on the `exists` function.

In [ ]:
import os
import functools

# Refactored: single expand_tilde decorator (replaces expand_tilde_first_and_then
# and expand_tilde2 — same behaviour, one consistent name, uses @wraps)
def expand_tilde(path_func):
    """Decorator: expand ~ in the first positional argument before calling path_func."""
    @functools.wraps(path_func)
    def wrapped(first_component, *parts):
        return path_func(os.path.expanduser(first_component), *parts)
    return wrapped

def countit(fn):
    counter = 0
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        nonlocal counter
        counter += 1
        result = fn(*args, **kwargs)
        print(f"call #{counter} to {fn.__name__}")
        return result
    return wrapper

# From refactored pathz/__init__.py:
@expand_tilde
@countit
def exists(path):
    """Check if path exists.  Returns False for broken symbolic links."""
    try:
        os.stat(path)
    except (OSError, ValueError):
        return False
    return True

join    = expand_tilde(countit(os.path.join))
abspath = expand_tilde(countit(os.path.abspath))

# Decorator stacking order:
#   Applied bottom-up: exists = expand_tilde(countit(raw_exists))
#   Called top-down:   expand_tilde runs first → expands ~ → passes to countit wrapper

print(exists("~/"))            # True
print(exists("~/nonexistent")) # False
print(join("~/repos", "github", "user", "repo"))

## 4.6 `functools.wraps` — Why It Matters

In [ ]:
import functools

def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper  # no @wraps → metadata lost

def good_decorator(func):
    @functools.wraps(func)  # copies __name__, __doc__, __module__, etc.
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def my_func():
    """Does something important."""
    pass

broken = bad_decorator(my_func)
fixed = good_decorator(my_func)

print(f"bad:   __name__={broken.__name__!r}, __doc__={broken.__doc__!r}")
print(f"good:  __name__={fixed.__name__!r},  __doc__={fixed.__doc__!r}")

## 4.7 Common Built-in Decorators

In [ ]:
from dataclasses import dataclass
from functools import cached_property, lru_cache

# --- @staticmethod / @classmethod ---
class MathHelper:
    @staticmethod
    def add(a, b):        # no self/cls — pure utility
        return a + b

    @classmethod
    def from_string(cls, s):  # cls = MathHelper
        a, b = map(int, s.split("+"))
        return cls.add(a, b)

print(MathHelper.add(3, 4))
print(MathHelper.from_string("10+5"))

# --- @property ---
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):        # accessed like an attribute, not a call
        import math
        return math.pi * self.radius ** 2

c = Circle(5)
print(f"area = {c.area:.2f}")   # no () needed

# --- @lru_cache ---
@lru_cache(maxsize=128)
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

print(fib(40))
print(fib.cache_info())

---
# Summary

```
COURSE CONCEPT MAP
══════════════════════════════════════════════════════════════════

  FUNCTIONS                  MODULES                 PACKAGES
  ─────────────────          ─────────────────       ─────────────────
  def / lambda               import styles           __init__.py
  parameters:                __name__                __main__.py
    positional               if __name__==…          relative imports
    keyword                  sys.path / sys.modules  pyproject.toml
    *args / **kwargs         module attributes       entry points
    defaults                 introspection           pipx / uv
  return values
  type hints
  LEGB scope

  DECORATORS & CLOSURES
  ─────────────────────────────────────────────────────────────────
  closures: capture variables from enclosing scope (nonlocal)
  decorators: wrap functions — @syntax = syntactic sugar
  decorator factories: parametrize decorators (3 levels deep)
  stack decorators: applied bottom-up, runs top-down
  functools.wraps: preserve function metadata
  built-ins: @property, @staticmethod, @classmethod, @lru_cache

  COURSE PROJECT: reponow + pathz
  ─────────────────────────────────────────────────────────────────
  parse_url()         — regex, URL parsing, tuple return
  clone_repo()        — subprocess, argparse
  expand_tilde2       — decorator + lambda closure
  countit             — decorator + nonlocal counter
  @expand_tilde2      ← stacked on ─┐
  @countit                           ├── exists(), join(), abspath()
  pytest parametrize  — table-driven tests
══════════════════════════════════════════════════════════════════
```

## Key Takeaways

| Concept | When to use |
|---|---|
| `*args / **kwargs` | When you don't know how many arguments a caller will pass |
| `keyword-only params (*)` | Force callers to be explicit for safety/readability |
| Modules | Logical grouping of related functions/classes in a `.py` file |
| Packages | Collection of modules with a public API via `__init__.py` |
| Closures | State without a class; factory functions |
| Decorators | Cross-cutting concerns — logging, timing, caching, auth |
| `@functools.wraps` | Always use inside decorators to preserve metadata |